In [1]:
# ======================================================
# COMBINED ML + STRATEGY BACKTESTING
# ======================================================

import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from xgboost import XGBClassifier
import joblib

# ------------------------------------------------------
# ENV SETUP
# ------------------------------------------------------
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------
# INTERNAL IMPORTS
# ------------------------------------------------------
from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import (
    DEFAULT_TICKERS,
    TRAIN_START,
    TRAIN_END,
    TEST_START,
    TEST_END,
)

# ------------------------------------------------------
# LOGGING
# ------------------------------------------------------
print("=" * 80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("=" * 80)
print(f"Testing Period : {TEST_START} → {TEST_END}")
print("=" * 80)


COMBINED ML + STRATEGY BACKTESTING
Testing Period : 2024-01-01 → 2024-12-31


In [2]:
# ======================================================
# SCALPING STRATEGY SIGNALS (CLEAN & REGIME-AWARE)
# ======================================================
def add_scalping_signals(data):
    df = data.copy()

    # -------------------------------
    # CORE INDICATORS
    # -------------------------------
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = 100 - (100 / (1 + rs))

    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - macd_signal

    # Volatility regime
    returns = df["Close"].pct_change()
    vol = returns.rolling(10).std()
    high_vol = vol > vol.rolling(50).mean()

    # -------------------------------
    # BUY CONDITIONS
    # -------------------------------
    buy_trend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = df["RSI"] < 45
    buy_macd = (macd > 0) & (macd_hist > 0)

    buy_signal = buy_trend & high_vol & (buy_rsi | buy_macd)

    # -------------------------------
    # SELL CONDITIONS
    # -------------------------------
    sell_trend = (df["Close"] < sma_20) & (sma_20 < sma_50)
    sell_rsi = df["RSI"] > 55
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = sell_trend & high_vol & (sell_rsi | sell_macd)

    # -------------------------------
    # FINAL SIGNAL (NO CONFLICT)
    # -------------------------------
    df["strategy_signal"] = 0
    df.loc[buy_signal, "strategy_signal"] = 1
    df.loc[sell_signal, "strategy_signal"] = -1

    return df

# ======================================================
# FEATURE ENGINEERING (RETURN-ALIGNED)
# ======================================================
def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    # -------------------------------
    # RETURNS
    # -------------------------------
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # -------------------------------
    # TREND FEATURES
    # -------------------------------
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # -------------------------------
    # PRICE ACTION
    # -------------------------------
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]
    df["body_abs"] = df["body_pct"].abs()

    # -------------------------------
    # VOLATILITY REGIME
    # -------------------------------
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # -------------------------------
    # RSI (0–1) — REUSED
    # -------------------------------
    df["RSI"] = df["RSI"] / 100.0

    # -------------------------------
    # VOLUME
    # -------------------------------
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # -------------------------------
    # TARGET (DIRECTION + COST AWARE)
    # -------------------------------
    future_return = df["Close"].shift(-horizon) / df["Close"] - 1
    df["target"] = ((future_return - cost) > 0).astype(int)

    df.dropna(inplace=True)
    return df


In [3]:
# ======================================================
# DATA LOADING & PREPARATION
# ======================================================
ticker = DEFAULT_TICKERS[0]

print("\n" + "=" * 80)
print(f"ANALYZING {ticker}")
print("=" * 80)

# ------------------------------------------------------
# Load raw OHLCV
# ------------------------------------------------------
raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)

# ------------------------------------------------------
# APPLY STRATEGY + FEATURES ON FULL DATA
# (prevents indicator warm-up bugs)
# ------------------------------------------------------
data_with_signals = add_scalping_signals(cleaned_data)
data_with_features = add_basic_features(data_with_signals)

# ------------------------------------------------------
# TIME-BASED SPLIT (POST FEATURE ENGINEERING)
# ------------------------------------------------------
train_data, test_data = split_data_by_date(data_with_features)

print(f"Train data shape : {train_data.shape}")
print(f"Test data shape  : {test_data.shape}")



ANALYZING NIFTY BANK
2025-12-30 14:41:33 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Nihar\Documents\GitHub\oop\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-30 14:41:34 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-30 14:41:34 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-30 14:41:34 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-30 14:41:34 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-30 14:41:34 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-30 14:41:34 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-30 14:41:34 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 09:18:00
2025-12-30 14:41:

In [4]:
# ------------------------------------------------------
# Feature selection (NO STRATEGY LEAKAGE)
# ------------------------------------------------------

EXCLUDE_COLS = {
    "target", "Open", "High", "Low", "Close", "Volume",
    "strategy_signal"
}

feature_cols = [c for c in train_data.columns if c not in EXCLUDE_COLS]

X_train = train_data[feature_cols]
y_train = train_data["target"]

X_test = test_data[feature_cols]
y_test = test_data["target"]

print(f"Features used: {len(feature_cols)}")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")


# ------------------------------------------------------
# Time-based validation split (NO leakage)
# ------------------------------------------------------
split_idx = int(0.8 * len(X_train))

X_tr, X_val = X_train.iloc[:split_idx], X_train.iloc[split_idx:]
y_tr, y_val = y_train.iloc[:split_idx], y_train.iloc[split_idx:]

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=3,              # ↓ shallower trees
    learning_rate=0.03,
    subsample=0.6,
    colsample_bytree=0.6,
    min_child_weight=40,      # ↑ stronger regularization
    gamma=0.2,
    reg_alpha=0.3,
    reg_lambda=1.5,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    verbosity=0
)

print("\nTraining XGBoost model...")

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_val_prob = xgb_model.predict_proba(X_val)[:, 1]
y_test_prob = xgb_model.predict_proba(X_test)[:, 1]

best_threshold = 0.5
best_score = -np.inf

baseline = y_val.mean()

for t in np.arange(0.40, 0.70, 0.02):
    preds = (y_val_prob > t).astype(int)
    trades = preds.sum()

    if trades < 30 or trades > 0.35 * len(preds):
        continue

    win_rate = y_val[preds == 1].mean() if trades > 0 else 0
    edge = win_rate - baseline

    # Expectancy proxy (better than win-rate alone)
    expectancy_score = edge * trades / len(preds)

    if expectancy_score > best_score:
        best_score = expectancy_score
        best_threshold = t

y_test_pred = (y_test_prob > best_threshold).astype(int)

print(f"\n✓ ML Threshold : {best_threshold:.2f}")
print(f"✓ Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"✓ Test AUC     : {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"✓ Test F1      : {f1_score(y_test, y_test_pred, zero_division=0):.4f}")


Features used: 13
X_train: (791543, 13)
X_test : (80907, 13)

Training XGBoost model...

✓ ML Threshold : 0.40
✓ Test Accuracy: 0.7013
✓ Test AUC     : 0.6038
✓ Test F1      : 0.2009
